# DATA304 · ColBERT + BM25 후보 합집합과 Cross-Encoder 점수 결합

각 VAL 쿼리에서 저장된 **ColBERT 상위 100개**와 전체 카탈로그에서 새로 찾은 **BM25 상위 100개**를 상품 ID 기준으로 합친다. 기본값은 합집합 전체(최대 200개)를 기존 **Full 5 Cross-Encoder**로 채점하고 최종 **Top-100**을 정한다. 과제의 주 지표는 **Recall@10**이다.

같은 쿼리의 재정렬 후보에서 Cross logit과 BM25 원점수를 **각각 min–max 정규화**한 뒤 다음 점수로 내림차순 정렬한다.

$$S_{\mathrm{final}}(q,d)=(1-\alpha)\widehat S_{\mathrm{Cross}}(q,d)+\alpha\widehat S_{\mathrm{BM25}}(q,d)$$

| $\alpha$ | Cross 가중치 | BM25 가중치 |
|---:|---:|---:|
| 0.25 | 0.75 | 0.25 |
| 0.50 | 0.50 | 0.50 |
| 0.75 | 0.25 | 0.75 |

기존 SageMaker 프로젝트의 `conda_pytorch` 커널에서 **Run All**을 실행한다. 기본 검증 범위는 직전 Full 5 실험과 같은 350개다. 빠른 점검은 첫 셀에서 `VAL_LIMIT = 10`으로 설정한다. **Cross 입력을 100개로 제한하는 비교**는 `RERANK_LIMIT = 100`으로 실행한다. 이때 양쪽의 1위, 양쪽의 2위 순서로 번갈아 보면서 서로 다른 상품 100개를 고른다. 두 설정은 결과를 별도 폴더에 저장한다.

In [ ]:
from collections import Counter
from contextlib import nullcontext
from itertools import zip_longest
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from sklearn.feature_extraction import DictVectorizer
from torch import nn
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
VAL_LIMIT = 350                 # None이면 기존 Mixed 실행의 VAL 전체
ALPHAS = (0.25, 0.50, 0.75)     # BM25 가중치
BM25_TOPK = 100
RERANK_LIMIT = None             # None: 합집합 전체 / 100: 합집합에서 최대 100개 선발
MIXED_RUN_ID = None             # 설정이 일치하는 실행이 여럿일 때만 폴더명 지정
INFER_BATCH_SIZE = None         # None이면 기존 cross_infer_batch

RUN_DIR = PROJECT_DIR / 'data304_outputs'
MIXED_ROOT = RUN_DIR / 'cross_hard4_mixed4_lr5e-6'
COLBERT_EPOCH, CROSS_EPOCH = 5, 3
SEARCH = dict(ann_per_token=1024, ann_nprobe=32, ann_pool=900, candidate_topk=100)
FINAL_TOPK = 100
K1, B, EPSILON = 1.5, 0.75, 0.25  # 기존 rank_bm25.BM25Okapi의 기본 설정
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
if type(BM25_TOPK) is not int or BM25_TOPK < 1:
    raise ValueError('BM25_TOPK에 양의 정수를 지정하세요.')
if RERANK_LIMIT is not None and (type(RERANK_LIMIT) is not int or RERANK_LIMIT < 1):
    raise ValueError('RERANK_LIMIT은 None 또는 양의 정수여야 합니다.')
ALPHAS = tuple(float(a) for a in ALPHAS)
if not ALPHAS or len(set(ALPHAS)) != len(ALPHAS) or any(not 0 <= a <= 1 for a in ALPHAS):
    raise ValueError('ALPHAS에는 0~1 범위의 서로 다른 수를 지정하세요.')

## 1. 기존 VAL과 ColBERT 후보 복원

Mixed Cross-Encoder epoch 3과 ColBERT epoch 5 실행의 설정, 카탈로그 순서, VAL 쿼리 순서를 대조한다. 각 쿼리의 기존 상위 100개는 ANN 900개를 MaxSim으로 재정렬한 순서다.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def signature(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(encoded).hexdigest()


def file_sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def find_mixed_run():
    parent = MIXED_ROOT / f'colbert_epoch_{COLBERT_EPOCH:02d}'
    manifests = ([parent / MIXED_RUN_ID / 'settings.json'] if MIXED_RUN_ID is not None
                 else sorted(parent.glob('*/settings.json')))
    matches = []
    for path in manifests:
        if not path.is_file():
            continue
        settings = read_json(path)
        training, mining = settings['training'], settings['mining']
        epoch_dir = path.parent / f'epoch_{CROSS_EPOCH:02d}'
        weights = epoch_dir / 'checkpoints' / 'cross_mixed.pt'
        if (weights.is_file()
                and training['colbert_epoch'] == COLBERT_EPOCH
                and CROSS_EPOCH in training['target_epochs']
                and (training['hard_count'], training['original_count'], training['original_mode']) == (4, 4, 'mixed')
                and training['lr'] == 5e-6
                and (training['config']['cross_tokens'], training['config']['cross_tokens_per_review']) == (448, 58)
                and all(mining[key] == value for key, value in SEARCH.items())
                and training['mining'] == signature(mining)):
            matches.append((path.parent, settings, weights))
    if not matches:
        raise FileNotFoundError(f'Mixed epoch 3의 settings.json과 epoch_03/checkpoints/cross_mixed.pt를 확인하세요. {parent}')
    if len(matches) > 1:
        names = [folder.name for folder, _, _ in matches]
        raise ValueError(f'조건이 같은 실행이 여러 개입니다. 첫 셀의 MIXED_RUN_ID에 폴더명을 지정하세요. {names}')
    return matches[0]


SOURCE_RUN, SAVED, CROSS_WEIGHTS = find_mixed_run()
MINING = SAVED['mining']
TRAIN_CONFIG = SAVED['training']['config']
TRAIN_LENGTH = TRAIN_CONFIG['cross_tokens']
PER_REVIEW = TRAIN_CONFIG['cross_tokens_per_review']
CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
for path, expected_hash in ((CATALOG_PATH, MINING['index']['catalog']),
                            (GOLD_PATH, MINING['gold']), (SPLIT_PATH, MINING['split'])):
    if file_sha(path) != expected_hash:
        raise ValueError(f'후보를 만들 때 사용한 파일과 내용이 다릅니다. {path}')

project_config = dict(TRAIN_CONFIG)
for config_path in (RUN_DIR / 'aspect_vocabulary_and_config.json', RUN_DIR / 'run_manifest.json'):
    if config_path.is_file():
        project_config.update(read_json(config_path)['config'])
BATCH = project_config.get('cross_infer_batch') if INFER_BATCH_SIZE is None else INFER_BATCH_SIZE
if type(BATCH) is not int or BATCH < 1:
    raise ValueError('기존 cross_infer_batch를 확인하거나 첫 셀에서 INFER_BATCH_SIZE에 양의 정수를 지정하세요.')

saved_ids = SAVED['val_query_ids']
split_ids = read_json(SPLIT_PATH)['val_query_ids']
if not saved_ids or saved_ids != split_ids[:len(saved_ids)] or len(set(saved_ids)) != len(saved_ids):
    raise ValueError('저장한 Mixed VAL의 쿼리와 split.json의 순서를 확인하세요.')
if VAL_LIMIT is not None and (type(VAL_LIMIT) is not int or not 1 <= VAL_LIMIT <= len(saved_ids)):
    raise ValueError(f'VAL_LIMIT은 1~{len(saved_ids)} 또는 None으로 지정하세요.')
VAL_IDS = saved_ids[:VAL_LIMIT]
gold_rows = {r['query_id']: r for r in read_json(GOLD_PATH)}
VAL = [gold_rows[qid] for qid in VAL_IDS]
del gold_rows


def metadata_text(item):
    fields = [('title', 'Product'), ('store', 'Store'), ('features', 'Features'), ('description', 'Description')]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


catalog = read_json(CATALOG_PATH)
ITEM_IDS = [item['item_id'] for item in catalog]
ITEM_TO_IDX = {iid: i for i, iid in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(item) for item in catalog]
del catalog
if len(ITEM_IDS) != len(ITEM_TO_IDX) or any(row['item_id'] not in ITEM_TO_IDX for row in VAL):
    raise ValueError('카탈로그의 상품 ID와 VAL 정답을 확인하세요.')

In [ ]:
def load_val_candidates():
    mining_key = signature(MINING)[:20]
    split_key = 'val_' + signature(saved_ids)[:16]  # 일부만 평가해도 원래 350건 캐시를 읽는다.
    roots = []
    for field, fallback in [('candidate_cache', MIXED_ROOT), ('previous_readonly_candidates', RUN_DIR / 'cross_hard_negative')]:
        recorded = Path(SAVED[field])
        if recorded.name != mining_key:
            raise ValueError(f'검색 설정과 후보 캐시의 식별자가 다릅니다. {recorded}')
        # 프로젝트 폴더를 옮겼을 때도 새 경로에서 같은 캐시를 찾는다.
        roots.extend([fallback / 'candidate_cache' / mining_key / split_key, recorded / split_key])
    roots = list(dict.fromkeys(roots))
    shards = {}
    for root in roots:
        for shard in sorted(root.glob('*.npz')):
            shards.setdefault(shard.name, shard)  # 원래 학습 코드도 mixed 캐시를 우선한다.

    n = len(VAL_IDS)
    topk = min(SEARCH['candidate_topk'], len(ITEM_IDS))
    candidates = np.empty((n, topk), dtype=np.int32)
    ranks = np.empty(n, dtype=np.int32)
    filled = np.zeros(n, dtype=bool)
    for path in shards.values():
        start, end = map(int, path.stem.split('_'))
        if not 0 <= start < end <= len(saved_ids):
            raise ValueError(f'VAL 범위가 잘못된 후보 묶음입니다. {path}')
        if start >= n:
            continue
        stop = min(n, end)
        with np.load(path, allow_pickle=False) as shard:
            docs, ann = shard['candidates'], shard['ann_ranks']
            if (not np.array_equal(shard['query_ids'], np.asarray(saved_ids[start:end], dtype=str))
                    or docs.shape != (end - start, topk) or ann.shape != (end - start,)
                    or docs.dtype.kind not in 'iu' or ann.dtype.kind not in 'iu'
                    or np.any((docs < 0) | (docs >= len(ITEM_IDS)))
                    or np.any((ann < 0) | (ann > SEARCH['ann_pool']))):
                raise ValueError(f'쿼리 순서, 후보 또는 ANN 순위가 잘못되었습니다. {path}')
            docs, ann = docs[:stop - start], ann[:stop - start]
            occupied = filled[start:stop]
            if (np.any(candidates[start:stop][occupied] != docs[occupied])
                    or np.any(ranks[start:stop][occupied] != ann[occupied])):
                raise ValueError(f'같은 쿼리에 서로 다른 후보가 저장되어 있습니다. {path}')
            candidates[start:stop], ranks[start:stop] = docs, ann
            filled[start:stop] = True
    if not filled.all():
        i = int(np.flatnonzero(~filled)[0])
        raise FileNotFoundError(f'기존 Mixed VAL 후보에서 {VAL_IDS[i]}가 누락되었습니다. npz 폴더: {roots}')
    if np.any(np.diff(np.sort(candidates, axis=1), axis=1) == 0):
        raise ValueError('한 쿼리의 상위 후보에 중복 상품이 있습니다.')
    return candidates, ranks


CANDIDATES, ANN_RANKS = load_val_candidates()
print(f'Cross 가중치 {CROSS_WEIGHTS}\nVAL {len(VAL)}건 × 후보 {CANDIDATES.shape[1]}개 | {SEARCH}')

## 2. Full 5 입력과 BM25 색인

Cross 쪽은 리뷰마다 앞 58토큰까지 읽고 다섯 개를 이어 붙인다. 상품과 함께 인코딩할 때의 상한은 448토큰이다. BM25도 기존 프로젝트의 `joined_reviews` 문자열을 쿼리로 쓰고, 전체 카탈로그의 `Product / Store / Features / Description` 문자열을 문서로 쓴다. 소문자로 바꾼 뒤 `[a-z0-9]+`로 토큰을 추출한다.

BM25의 문서 빈도와 평균 문서 길이는 **전체 카탈로그**에서 계산한다. 단어별로 그 단어를 포함한 상품과 점수 기여분을 저장해 반복 검색을 빠르게 한다. ColBERT만 찾아낸 상품에도 실제 BM25 점수를 부여한다. [프로젝트가 사용한 BM25Okapi의 점수식](https://github.com/dorianbrown/rank_bm25/blob/master/rank_bm25.py)

In [ ]:
backbone = TRAIN_CONFIG['backbone']
config = AutoConfig.from_pretrained(backbone)
config.num_labels = 1
TOKENIZER = AutoTokenizer.from_pretrained(backbone, use_fast=True)
if TRAIN_LENGTH > min(config.max_position_embeddings, TOKENIZER.model_max_length):
    raise ValueError('저장된 입력 길이가 모델 또는 토크나이저의 상한을 넘습니다.')
tokenizer_hash = hashlib.sha256(TOKENIZER.backend_tokenizer.to_str().encode()).hexdigest()


def joined_reviews(row):
    reviews = row['reviews']
    if len(reviews) != 5:
        raise ValueError(f"리뷰가 다섯 개여야 합니다. {row['query_id']}")
    chunks = []
    for number, review in enumerate(reviews, 1):
        ids = TOKENIZER.encode(review['text'].strip(), add_special_tokens=False,
                               truncation=True, max_length=PER_REVIEW)
        chunks.append(f'Review {number}: ' + TOKENIZER.decode(ids, skip_special_tokens=True))
    return ' [SEP] '.join(chunks)


QUERIES = [joined_reviews(row) for row in VAL]


class CrossReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = AutoModelForSequenceClassification.from_config(config)


def load_cross():
    model = CrossReranker()
    weights = torch.load(CROSS_WEIGHTS, map_location='cpu', weights_only=True, mmap=True)
    model.load_state_dict(weights, strict=True)
    return model.to(DEVICE).eval().requires_grad_(False).net


@torch.inference_mode()
def cross_scores(model, query, docs):
    scores = np.empty(len(docs), dtype=np.float32)
    for start in range(0, len(docs), BATCH):
        batch = docs[start:start + BATCH]
        encoded = TOKENIZER([query] * len(batch), [CAT_TEXTS[d] for d in batch], padding=True,
                            truncation='longest_first', max_length=TRAIN_LENGTH, return_tensors='pt')
        inputs = {key: value.to(DEVICE) for key, value in encoded.items()}
        with (torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()):
            logits = model(**inputs, return_dict=True).logits.reshape(-1)
        scores[start:start + len(batch)] = logits.float().cpu().numpy()
    if not np.isfinite(scores).all():
        raise ValueError('Cross-Encoder가 유한하지 않은 점수를 반환했습니다.')
    return scores

In [ ]:
def tokenize_lexical(text):
    return re.findall(r'[a-z0-9]+', text.lower())


def build_bm25(texts):
    # CSC의 열 하나가 한 단어를 포함한 상품 목록이다.
    vec = DictVectorizer(sort=False, dtype=np.float64)
    counts = (Counter(tokenize_lexical(text)) for text in texts)
    index = vec.fit_transform(counts).tocsc()
    lengths = np.asarray(index.sum(axis=1)).ravel()
    df = np.diff(index.indptr).astype(np.int64)
    n_docs = index.shape[0]
    idf = np.empty(len(df), dtype=np.float64)
    idf_sum = 0.0
    # 원래 BM25Okapi가 카탈로그에서 처음 본 단어 순서대로 IDF를 더하는 방식과 맞춘다.
    for column, freq in enumerate(df.tolist()):
        value = math.log(n_docs - freq + 0.5) - math.log(freq + 0.5)
        idf[column] = value
        idf_sum += value
    idf[idf < 0] = EPSILON * (idf_sum / len(idf))
    length_norm = K1 * (1 - B + B * lengths / lengths.mean())
    tf = index.data
    index.data = np.repeat(idf, df) * (tf * (K1 + 1) / (tf + length_norm[index.indices]))
    return vec.vocabulary_, index


def bm25_scores(query, vocabulary, index):
    scores = np.zeros(index.shape[0], dtype=np.float64)
    for term in tokenize_lexical(query):
        column = vocabulary.get(term)
        if column is not None:
            start, end = index.indptr[column:column + 2]
            scores[index.indices[start:end]] += index.data[start:end]
    return scores


# 프로젝트의 원래 BM25 패키지가 있으면, 작은 카탈로그 표본에서 원점수까지 대조한다.
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    BM25Okapi = None

if BM25Okapi is not None:
    sample = CAT_TEXTS[:min(128, len(CAT_TEXTS))]
    reference = BM25Okapi([tokenize_lexical(t) for t in sample], k1=K1, b=B, epsilon=EPSILON)
    vocab_check, index_check = build_bm25(sample)
    for probe in (QUERIES[0], sample[0]):
        np.testing.assert_allclose(bm25_scores(probe, vocab_check, index_check),
                                   reference.get_scores(tokenize_lexical(probe)), rtol=1e-10, atol=1e-10)
    del reference, vocab_check, index_check

## 3. 쿼리별 후보 합집합과 원점수 저장

BM25 순위는 전체 카탈로그를 검색해서 구한다. 기본 합집합에서는 ColBERT 후보를 먼저 놓고 BM25에만 있는 후보를 추가한다. 같은 상품은 한 번만 채점하고 이 점수를 세 α와 비교 기준이 공유한다. 동점은 이 후보 순서를 유지한다. BM25 검색 자체에서 동점이면 카탈로그 순서를 유지한다.

원점수는 10개 쿼리 단위로 저장한다. 같은 설정으로 재시작하면 완료한 묶음을 재사용하고, `ALPHAS`만 바꾸면 저장한 점수로 새 표를 만든다.

In [ ]:
def union_and_pool(colbert, lexical):
    union = list(dict.fromkeys([*colbert, *lexical]))
    if RERANK_LIMIT is None or len(union) <= RERANK_LIMIT:
        return union, union
    interleaved = (doc for pair in zip_longest(colbert, lexical)
                   for doc in pair if doc is not None)
    # 양쪽 동일 순위에서는 ColBERT를 먼저 본다. 점수 동점 처리에는 기본 합집합 순서를 쓴다.
    selected = set(list(dict.fromkeys(interleaved))[:RERANK_LIMIT])
    return union, [doc for doc in union if doc in selected]


def score_query(i, model, bm25_index):
    query = QUERIES[i]
    colbert = CANDIDATES[i].tolist()
    lexical_scores = bm25_scores(query, *bm25_index)
    if not np.isfinite(lexical_scores).all():
        raise ValueError(f'BM25 점수를 확인하세요. {VAL_IDS[i]}')
    lexical = np.argsort(-lexical_scores, kind='stable')[:BM25_TOPK].tolist()
    _, pool = union_and_pool(colbert, lexical)
    existing = set(colbert)
    extra = [d for d in pool if d not in existing]
    scored_docs = colbert + extra
    # 기존 Top-100의 추론 배치 구성을 보존하고, 추가된 상품만 이어서 채점한다.
    logits = np.concatenate([cross_scores(model, query, colbert), cross_scores(model, query, extra)])
    return {'query_id': str(VAL_IDS[i]), 'bm25_top': lexical, 'scored_docs': scored_docs,
            'cross': logits.tolist(), 'bm25': lexical_scores[scored_docs].tolist()}


def save_json(data, path):
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        staged.write_text(json.dumps(data, ensure_ascii=False, allow_nan=False), encoding='utf-8')
        staged.replace(path)


def save_csv(frame, path):
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        frame.to_csv(staged, index=False)
        staged.replace(path)


score_spec = {
    'version': 1, 'source_run': SOURCE_RUN.name, 'colbert_epoch': COLBERT_EPOCH, 'cross_epoch': CROSS_EPOCH,
    'checkpoint_sha256': file_sha(CROSS_WEIGHTS), 'mining': MINING, 'val_ids': VAL_IDS,
    'colbert_candidates_sha256': hashlib.sha256(CANDIDATES.tobytes()).hexdigest(),
    'backbone': backbone, 'tokenizer_sha256': tokenizer_hash, 'cross_tokens': TRAIN_LENGTH,
    'cross_tokens_per_review': PER_REVIEW, 'cross_input': 'full5', 'cross_batch': BATCH,
    'bm25_topk': BM25_TOPK, 'bm25_k1': K1, 'bm25_b': B, 'bm25_epsilon': EPSILON,
    'bm25_query': 'joined_reviews', 'lexical_pattern': '[a-z0-9]+', 'rerank_limit': RERANK_LIMIT,
    'pool_selection': 'union' if RERANK_LIMIT is None else 'rank_interleaving',
    'device': DEVICE.type, 'precision': 'bfloat16' if USE_BF16 else 'float32',
    'torch_version': str(torch.__version__), 'transformers_version': str(transformers.__version__),
    'numpy_version': np.__version__, 'sklearn_version': sklearn.__version__,
}
run_root = RUN_DIR / 'cross_bm25_union_val' / SOURCE_RUN.name / signature(score_spec)[:16]
parts_dir = run_root / 'scores'
parts_dir.mkdir(parents=True, exist_ok=True)
save_json({**score_spec, 'cross_weights': str(CROSS_WEIGHTS)}, run_root / 'settings.json')
records = []
model, bm25_index = None, None
chunk_size = 10
with tqdm(total=len(VAL), desc='VAL queries') as progress:
    for start in range(0, len(VAL), chunk_size):
        end = min(start + chunk_size, len(VAL))
        path = parts_dir / f'{start:04d}_{end:04d}.json'
        if path.is_file():
            part = read_json(path)
            if [r['query_id'] for r in part] != [str(qid) for qid in VAL_IDS[start:end]]:
                raise ValueError(f'저장한 원점수의 쿼리 순서가 다릅니다. {path}')
            progress.update(len(part))
        else:
            if bm25_index is None:
                print(f'BM25 색인 구성 | 카탈로그 {len(CAT_TEXTS):,}개', flush=True)
                bm25_index = build_bm25(CAT_TEXTS)
                model = load_cross()
            part = []
            for i in range(start, end):
                part.append(score_query(i, model, bm25_index))
                progress.update(1)
            save_json(part, path)
        records.extend(part)

## 4. 세 가중치 비교

**쿼리별 재정렬 후보 안에서** 두 점수에 각각 $\hat s=(s-\min s)/(\max s-\min s)$를 적용한다. 한 모델이 모든 후보에 같은 점수를 주면 그 모델의 정규화 점수는 모두 0이다. 정답 ID는 순위를 만든 뒤 평가할 때 사용한다.

- **ColBERT → Cross**는 기존 Full 5 기준선이다. **재정렬 후보 → Cross**와 비교하면 후보를 바꾼 효과를 볼 수 있다.
- **재정렬 후보 → Cross**와 세 **Hybrid** 행은 같은 상품 집합을 사용하므로 점수 결합 효과를 비교할 수 있다. **BM25 단독** 행은 단어 검색 자체의 기준선이다.
- P@1, Recall@10, MRR@100의 분모는 전체 VAL이다. `candidate_recall`은 재정렬 전에 정답을 포함한 비율이고, `recall_at_100`은 최종 100위 안에 정답이 든 비율이다. `gained_at_10`과 `lost_at_10`은 기존 ColBERT → Cross 대비 변동이다.
- `stage_summary.csv`는 전체 비교표, `query_diagnostics.csv`는 쿼리별 정답 순위, `candidate_diagnostics.csv`는 후보 출처와 겹침을 담는다. `hybrid_top100.csv`에는 세 α별 최종 상품 ID가 순서대로 들어간다.

In [ ]:
def minmax(scores):
    x = np.asarray(scores, dtype=np.float64)
    low, high = x.min(), x.max()
    return (x - low) / (high - low) if high > low else np.zeros_like(x)


def descending(docs, scores):
    return np.asarray(docs, dtype=np.int64)[np.argsort(-scores, kind='stable')]


def evaluate_order(qid, gold_id, gold_idx, method, alpha, ranked):
    pos = np.flatnonzero(np.asarray(ranked) == gold_idx)
    rank = int(pos[0]) + 1 if len(pos) else None
    at_100 = rank is not None and rank <= FINAL_TOPK
    return dict(query_id=qid, gold_item_id=str(gold_id), method=method, alpha_bm25=alpha,
                n_candidates=len(ranked), gold_in_candidates=rank is not None,
                gold_rank_before_cut=rank, gold_rank_in_top100=rank if at_100 else None,
                top1_item_id=str(ITEM_IDS[ranked[0]]), top1_hit=rank == 1,
                top10_hit=rank is not None and rank <= 10, top100_hit=at_100,
                rr_at_100=1.0 / rank if at_100 else 0.0)


BASE, CROSS_POOL, BM_ONLY = 'ColBERT → Cross', '재정렬 후보 → Cross', 'BM25 단독'
hybrid_names = {a: f'Hybrid α={a}' for a in ALPHAS}
source_names = {(True, True): 'both', (True, False): 'colbert_only',
                (False, True): 'bm25_only', (False, False): 'neither'}
detail_rows, candidate_rows, predictions = [], [], []
for i, rec in enumerate(records):
    qid, gold_id = str(VAL_IDS[i]), VAL[i]['item_id']
    gold_idx = ITEM_TO_IDX[gold_id]
    colbert, lexical = CANDIDATES[i].tolist(), rec['bm25_top']
    union, pool = union_and_pool(colbert, lexical)
    docs = rec['scored_docs']
    colset, lexset = set(colbert), set(lexical)
    expected_docs = colbert + [d for d in pool if d not in colset]
    cross = np.asarray(rec['cross'], dtype=np.float64)
    bm = np.asarray(rec['bm25'], dtype=np.float64)
    if (qid != rec['query_id'] or docs != expected_docs or len(lexset) != len(lexical)
            or len(lexical) != min(len(ITEM_IDS), BM25_TOPK)
            or any(d < 0 or d >= len(ITEM_IDS) for d in lexical)
            or cross.shape != (len(docs),) or bm.shape != cross.shape
            or not np.isfinite(cross).all() or not np.isfinite(bm).all()):
        raise ValueError(f'후보 ID와 원점수 정렬을 확인하세요. {qid}')
    source = source_names[gold_idx in colset, gold_idx in lexset]
    candidate_rows.append(dict(query_id=qid, gold_item_id=str(gold_id), gold_source=source,
        n_colbert=len(colbert), n_bm25=len(lexical), n_overlap=len(colset & lexset),
        n_union=len(union), n_rerank=len(pool), gold_in_colbert=gold_idx in colset,
        gold_in_bm25=gold_idx in lexset, gold_in_union=gold_idx in union, gold_in_rerank=gold_idx in pool))

    positions = {d: j for j, d in enumerate(docs)}
    indices = [positions[d] for d in pool]
    cross_pool, bm_pool = cross[indices], bm[indices]
    cross_hat, bm_hat = minmax(cross_pool), minmax(bm_pool)
    rankings = [(BASE, None, descending(colbert, cross[:len(colbert)])),
                (CROSS_POOL, None, descending(pool, cross_pool)),
                (BM_ONLY, None, lexical)]
    for alpha in ALPHAS:
        fused = (1.0 - alpha) * cross_hat + alpha * bm_hat
        ranked = descending(pool, fused)
        rankings.append((hybrid_names[alpha], alpha, ranked))
        top_ids = {f'item_id_{rank}': str(ITEM_IDS[d]) for rank, d in enumerate(ranked[:FINAL_TOPK], 1)}
        predictions.append(dict(query_id=qid, alpha_bm25=alpha, **top_ids))
    for name, alpha, ranked in rankings:
        detail_rows.append(evaluate_order(qid, gold_id, gold_idx, name, alpha, ranked))

detail = pd.DataFrame(detail_rows)
for col in ('gold_rank_before_cut', 'gold_rank_in_top100'):
    detail[col] = detail[col].astype('Int64')
candidates = pd.DataFrame(candidate_rows)
base10 = detail.loc[detail.method == BASE].set_index('query_id', verify_integrity=True)['top10_hit']
pool10 = detail.loc[detail.method == CROSS_POOL].set_index('query_id', verify_integrity=True)['top10_hit']
detail['baseline_top10_hit'] = detail['query_id'].map(base10)
summary_rows = []
for method, frame in detail.groupby('method', sort=False):
    hit10 = frame['top10_hit']
    previous_hit = frame['baseline_top10_hit']
    pool_hit = frame['query_id'].map(pool10)
    summary_rows.append(dict(
        method=method, alpha_bm25=frame['alpha_bm25'].iloc[0], n_queries=len(frame),
        mean_candidates=frame['n_candidates'].mean(), candidate_hits=int(frame['gold_in_candidates'].sum()),
        candidate_recall=frame['gold_in_candidates'].mean(),
        top1_hits=int(frame['top1_hit'].sum()), precision_at_1=frame['top1_hit'].mean(),
        top10_hits=int(hit10.sum()), recall_at_10=hit10.mean(), mrr_at_100=frame['rr_at_100'].mean(),
        recall_at_100=frame['top100_hit'].mean(), gained_at_10=int((hit10 & ~previous_hit).sum()),
        lost_at_10=int((~hit10 & previous_hit).sum()),
        delta_r10_vs_baseline_pp=100 * (hit10.mean() - previous_hit.mean()),
        delta_r10_vs_pool_cross_pp=100 * (hit10.mean() - pool_hit.mean()),
    ))
summary = pd.DataFrame(summary_rows)

report = run_root / ('alphas_' + '_'.join(str(a) for a in ALPHAS))
report.mkdir(exist_ok=True)
save_csv(summary, report / 'stage_summary.csv')
save_csv(detail, report / 'query_diagnostics.csv')
save_csv(candidates, report / 'candidate_diagnostics.csv')
save_csv(pd.DataFrame(predictions), report / 'hybrid_top100.csv')
save_json(dict(alphas_bm25=list(ALPHAS), normalization='per-query min-max on rerank candidates',
               constant_scores=0, final_topk=FINAL_TOPK, metric='recall_at_10'), report / 'fusion_settings.json')

display(summary[['method', 'alpha_bm25', 'mean_candidates', 'candidate_recall', 'top10_hits',
                 'recall_at_10', 'delta_r10_vs_baseline_pp', 'delta_r10_vs_pool_cross_pp',
                 'gained_at_10', 'lost_at_10', 'precision_at_1', 'mrr_at_100']].round(4))
print(f"정답 포함 쿼리 / {len(candidates)} | ColBERT {candidates.gold_in_colbert.sum()}"
      f" | BM25 {candidates.gold_in_bm25.sum()} | 합집합 {candidates.gold_in_union.sum()}"
      f" | 실제 재정렬 후보 {candidates.gold_in_rerank.sum()}")
hybrid = summary.loc[summary['alpha_bm25'].notna()]
best = hybrid.loc[hybrid['top10_hits'] == hybrid['top10_hits'].max()]
print(f'{len(ALPHAS)}개 조합 중 최고 VAL Recall@10', f"{best['recall_at_10'].iloc[0]:.2%}",
      '| BM25 가중치', ', '.join(f'{a:g}' for a in best['alpha_bm25']))
print('결과 폴더', report)